# 5. Sample-Level Evaluation

The most direct way to evaluate a detector is to compare its per-sample labels against a human rater's, sample by
sample — no need to construct Event objects first. `peyes.sample_metrics` treats both sequences as sequences of
`EventLabelEnum` (or raw ints/strings) and gives you agreement statistics, from a full confusion matrix down to
single-number scores.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()
ground_truth = d["raters"]["RA"]

detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
prediction, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)

## Confusion matrix and label counts

`confusion_matrix` rows are ground-truth labels, columns are predicted labels (0=UNDEFINED, 1=FIXATION,
2=SACCADE, 3=PSO, 4=SMOOTH_PURSUIT, 5=BLINK):

In [2]:
peyes.sample_metrics.confusion_matrix(ground_truth, prediction)

Prediction,0,1,2,3,4,5
Ground Truth,,,,,,
0,4,1,20,0,0,0
1,3,984,16,0,0,0
2,0,11,233,0,0,0
3,0,19,62,0,0,0
4,0,1266,56,0,0,0
5,6,4,52,0,0,47


Engbert never predicts PSO (3) or smooth pursuit (4) — it isn't designed to detect them — so the ground
truth's rows 3 and 4 are entirely misclassified as fixation/saccade. `label_counts` and `transition_matrix` describe
a single sequence on its own:

In [3]:
peyes.sample_metrics.label_counts(ground_truth), peyes.sample_metrics.label_counts(prediction)

(0      25
 1    1003
 2     244
 3      81
 4    1322
 5     109
 Name: count, dtype: int64,
 0      13
 1    2285
 2     439
 3       0
 4       0
 5      47
 Name: count, dtype: int64)

## Global agreement metrics

These compare the two full sequences with one number each, without singling out any one label:

In [4]:
for metric_fn in [peyes.sample_metrics.accuracy, peyes.sample_metrics.balanced_accuracy,
                   peyes.sample_metrics.cohen_kappa, peyes.sample_metrics.mcc, peyes.sample_metrics.complement_nld]:
    print(f"{metric_fn.__name__}: {metric_fn(ground_truth, prediction):.3f}")

accuracy: 0.455
balanced_accuracy: 0.421
cohen_kappa: 0.211
mcc: 0.332
complement_nld: 0.457


## Metrics for one label

`precision`, `recall`, `f1_score`, `d_prime`, and `criterion` all take a `pos_labels` argument, treating that label
(or set of labels) as "positive" and everything else as "negative" — a per-label signal-detection view:

In [5]:
fixation = peyes.parse_label("fixation")
print(f"precision: {peyes.sample_metrics.precision(ground_truth, prediction, pos_labels=fixation):.3f}")
print(f"recall:    {peyes.sample_metrics.recall(ground_truth, prediction, pos_labels=fixation):.3f}")
print(f"f1:        {peyes.sample_metrics.f1_score(ground_truth, prediction, pos_labels=fixation):.3f}")
print(f"d-prime:   {peyes.sample_metrics.d_prime(ground_truth, prediction, pos_labels=fixation):.3f}")
print(f"criterion: {peyes.sample_metrics.criterion(ground_truth, prediction, pos_labels=fixation):.3f}")

precision: 0.431
recall:    0.981
f1:        0.599
d-prime:   1.462
criterion: -1.345


`calculate` computes several metrics in one call, returning a dict when more than one is requested — useful
once you know which metrics you want:

In [6]:
peyes.sample_metrics.calculate(
    ground_truth, prediction, "accuracy", "cohen's_kappa", "recall", pos_labels=fixation,
)

{'accuracy': 0.45545977011494254,
 "cohen's_kappa": 0.2105580055654097,
 'recall': 0.9810568295114656}

## Across multiple trials

Real analyses rarely stop at one trial. Looping `sample_metrics` calls over several trials and collecting the
results into a `pandas.Series`/`DataFrame` is the same pattern as above, just repeated:

In [7]:
import pandas as pd

lund2013 = peyes.datasets.lund2013(directory="data", save=True, verbose=False)
rated_trials = lund2013.dropna(subset=["RA"])
trial_ids = sorted(rated_trials["trial_id"].unique())[:5]

scores = {}
for trial_id in trial_ids:
    trial = lund2013[lund2013["trial_id"] == trial_id]
    detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
    labels, _ = detector.detect(
        t=trial["t"].values, x=trial["x"].values, y=trial["y"].values,
        pixel_size_cm=trial["pixel_size"].values[0], viewer_distance_cm=trial["viewer_distance"].values[0],
    )
    scores[trial_id] = peyes.sample_metrics.balanced_accuracy(trial["RA"].values, labels)

pd.Series(scores, name="balanced_accuracy")

1    0.491597
2    0.522733
3    0.458667
4    0.470000
5    0.322581
Name: balanced_accuracy, dtype: float64

## What's next

**[6 Events - Construction & Properties](./6%20Events%20-%20Construction%20%26%20Properties.ipynb)** — moving from
flat label arrays to `Event` objects, which the rest of the guide (event metrics, matching, alignment) builds on.